<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">هر سطر <bdi dir="ltr">Logits</bdi> باید هدف خودش را داشته باشد</h1>
<p style="text-align:right">درس 49 از 92 · نمایش <bdi dir="ltr">C</bdi>تایی چگونه <bdi dir="ltr">V</bdi> امتیاز می‌سازد؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">43-lm-head</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-01/43-lm-head.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">خروجی <bdi dir="ltr">Language-model head</bdi> و <bdi dir="ltr">Loss</bdi> را با حفظ تطابق <bdi dir="ltr">Batch/Time</bdi> بسازید.</p><p style="text-align:right">پیش‌نیاز: <bdi dir="ltr">Linear</bdi>، <bdi dir="ltr">Logits</bdi>، <bdi dir="ltr">Cross-Entropy</bdi> و هدف یک‌خانه‌جلو را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۴۵–۸۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">برای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">B=2</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">T=3</code>، هدفِ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(1,0)</code> پس از صاف‌کردن در کدام سطر است؟ اگر فقط <bdi dir="ltr">Target</bdi> را <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">transpose</code> کنیم، <bdi dir="ltr">Shape</bdi> خطا را آشکار می‌کند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch.nn import functional as F
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
model = MiniGPT(ModelConfig(7,6,8,2,1,0.)).eval()
hidden = torch.randn(2,3,8)
targets = torch.tensor([[1,2,3],[4,5,6]])
print('head weight:',model.language_model_head.weight.shape,'targets:',targets)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">prediction_loss(hidden, weight, targets)</code> زوج <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(logits, loss)</code> برگرداند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">weight</code> همان وزن <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(V,C)</code> بدون <bdi dir="ltr">Bias</bdi> است؛ <bdi dir="ltr">Logits</bdi> را خودتان ضرب کنید و برای <bdi dir="ltr">Loss</bdi>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">B</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">T</code> را در هر دو ورودی با ترتیب یکسان صاف کنید. پیش از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">F.cross_entropy</code>، <bdi dir="ltr">Softmax</bdi> نزنید.</p>
</div>

In [ ]:
def prediction_loss(hidden, weight, targets):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = prediction_loss(hidden,model.language_model_head.weight,targets)
    if result is None: return False
    logits,loss = result
    torch.testing.assert_close(logits,model.language_model_head(hidden))
    separate = torch.stack([F.cross_entropy(logits[b,t][None],targets[b,t][None]) for b in range(2) for t in range(3)]).mean()
    torch.testing.assert_close(loss,separate)
    x = torch.randn(1,4,3); w = torch.randn(5,3); y = torch.tensor([[0,1,2,3]])
    z,l = prediction_loss(x,w,y)
    torch.testing.assert_close(z,x@w.T)
    torch.testing.assert_close(l,F.cross_entropy(z.reshape(-1,5),y.reshape(-1)))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <bdi dir="ltr">Vocabulary</bdi> را از ۷ به ۱۰ افزایش دهید؛ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">C</code> ثابت است. اینجا فقط تعداد امتیازها و <bdi dir="ltr">Parameter</bdi>های <bdi dir="ltr">Head</bdi> را مقایسه می‌کنیم، نه کیفیت مدل‌های تصادفی تازه را.</p>
</div>

In [ ]:
from torch import nn
for V in (7,10):
    head = nn.Linear(8,V,bias=False)
    print('V, output shape, parameters:',V,head(hidden).shape,sum(p.numel() for p in head.parameters()))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">دادهٔ کنترل‌شدهٔ زیر برای هر سطر، امتیاز کلاس درست را بالا گذاشته است. شش کلاس دیگر هرکدام ۸ واحد امتیاز کمتر دارند؛ بنابراین <bdi dir="ltr">Loss</bdi> درست هر سطر حدود ۰٫۰۰۲ است. کد خراب <bdi dir="ltr">Target</bdi> را با ترتیب زمان-اول صاف می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">aligned_loss(logits,targets)</code> را اصلاح کنید.</p>
</div>

In [ ]:
fixture = torch.full((2,3,7),-4.)
fixture.scatter_(-1,targets[...,None],4.)
wrong = F.cross_entropy(fixture.reshape(-1,7),targets.T.reshape(-1))
correct = fixture.new_tensor(math.log1p(6*math.exp(-8.)))
print('wrong ordering/known expected loss:',wrong.item(),correct.item())
assert wrong.item() > 1.

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def aligned_loss(logits, targets):
    # TODO
    return None

In [ ]:
def test_repair():
    result = aligned_loss(fixture,targets)
    if result is None: return False
    torch.testing.assert_close(result,correct)
    assert result.item() < 0.01
    logits = torch.tensor([[[1.,2.],[3.,-1.]]])
    y = torch.tensor([[1,0]])
    torch.testing.assert_close(aligned_loss(logits,y),F.cross_entropy(logits[0],y[0]))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">language_model_head</code> و محاسبهٔ نهایی <bdi dir="ltr">Loss</bdi> در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> همین قرارداد را دارند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">hidden</code> واقعی مدل پس از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">final_norm</code> می‌آید؛ اینجا نمایش کوچکِ آماده دادیم تا خطای تطابق سطرها جدا دیده شود.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا <bdi dir="ltr">Loss</bdi> عددی و <bdi dir="ltr">Shape</bdi> درست برای تأیید جفت‌شدن هر پیش‌بینی با هدف درست کافی نیست؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-01/43-lm-head.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/43-lm-head.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>